# El modelo completo de llamadas al centro de atención de urgencia

**¡Ahora podemos actualizar el modelo para incluir la lógica final del centro de llamadas!**

Después de que un paciente ha hablado con un operador de llamadas, se clasifica su prioridad.  Se estima que el 40% de los pacientes requieren que una enfermera les devuelva la llamada.  Hay 10 enfermeras disponibles.  Una consulta de enfermería al paciente tiene una distribución uniforme con una duración de entre 10 y 20 minutos.

> ⏰ Algunos centros de llamadas funcionan las 24 horas del día, los 7 días de la semana, mientras que otros están abiertos durante un período de tiempo específico durante el día.  Por lo tanto, debemos hacer frente a **tanto sistemas terminantes como no terminantes**.

![imagen del modelo](img/full_model.png "Centro de llamadas de atención de urgencia")

**Modificaciones necesarias**

* Agregar nuevas variables predeterminadas para los parámetros de consulta de enfermería.
* Agregar nuevas variables de decisión a `Experiment` para el no. enfermeras y la distribución de consultas.
* Cree un segundo `simpy.Resource` llamado `nurses` y agréguelo al modelo de simulación.
* Crear el proceso de consulta de enfermería.
* Modificar la lógica de `service` para que se devuelva la llamada al % de pacientes.
* Recopilar resultados y estimar el tiempo de espera para una consulta de enfermería y utilización de enfermeras.
* **Bonificación:** Agregue un evento de período de calentamiento *opcional*.

## 1. Importaciones

In [1]:
import numpy as np
import pandas as pd
import simpy
import itertools

## 2. Variables, constantes y valores predeterminados a nivel de cuaderno

Un primer paso útil al configurar un modelo de simulación es definir el caso base o los parámetros tal como están.  Aquí crearemos un conjunto de valores constantes/predeterminados para nuestra clase `Experiment`, pero también podría considerar leerlos desde un archivo.

In [2]:
# default resources
N_OPERATORS = 13

# ##############################################################################
# MODIFICATION: number of nurses available
N_NURSES = 10
# ##############################################################################

# default mean inter-arrival time (exp)
MEAN_IAT = 60 / 100

## default service time parameters (triangular)
CALL_LOW = 5.0
CALL_MODE = 7.0
CALL_HIGH = 10.0

# ##############################################################################
# MODIFICATION: nurse defaults

# nurse uniform distribution parameters
NURSE_CALL_LOW = 10.0
NURSE_CALL_HIGH = 20.0

# probability of a callback (parameter of Bernoulli)
CHANCE_CALLBACK = 0.4

# sampling settings - we now need 4 streams
N_STREAMS = 4
DEFAULT_RND_SET = 0
# ##############################################################################

# Boolean switch to display simulation results as the model runs
TRACE = False

# run variables
RESULTS_COLLECTION_PERIOD = 1000

# ##############################################################################
# MODIFICATON: added a warm-up period, by default we will not use it.
WARM_UP_PERIOD = 0
# ##############################################################################

## 3. Clases de distribución

Definiremos dos clases de distribución adicionales (`Uniform` y `Bernoulli`) para encapsular la generación de números aleatorios, los parámetros y las semillas aleatorias utilizadas en el muestreo.  Echa un vistazo a cómo funcionan.

> Debería poder reutilizar estas clases en sus propios modelos de simulación.  En realidad no es mucho código, pero es útil para crear una base de código que puedas reutilizar con confianza en tus propios proyectos.

In [3]:
class Bernoulli:
    """
    Convenience class for the Bernoulli distribution.
    packages up distribution parameters, seed and random generator.

    The Bernoulli distribution is a special case of the binomial distribution
    where a single trial is conducted

    Use the Bernoulli distribution to sample success or failure.
    """

    def __init__(self, p, random_seed=None):
        """
        Constructor

        Params:
        ------
        p: float
            probability of drawing a 1

        random_seed: int | SeedSequence, optional (default=None)
            A random seed to reproduce samples.  If set to none then a unique
            sample is created.
        """
        self.rand = np.random.default_rng(seed=random_seed)
        self.p = p

    def sample(self, size=None):
        """
        Generate a sample from the exponential distribution

        Params:
        -------
        size: int, optional (default=None)
            the number of samples to return.  If size=None then a single
            sample is returned.

        Returns:
        -------
        float or np.ndarray (if size >=1)
        """
        return self.rand.binomial(n=1, p=self.p, size=size)

In [4]:
class Uniform:
    """
    Convenience class for the Uniform distribution.
    packages up distribution parameters, seed and random generator.
    """

    def __init__(self, low, high, random_seed=None):
        """
        Constructor

        Params:
        ------
        low: float
            lower range of the uniform

        high: float
            upper range of the uniform

        random_seed: int | SeedSequence, optional (default=None)
            A random seed to reproduce samples.  If set to none then a unique
            sample is created.
        """
        self.rand = np.random.default_rng(seed=random_seed)
        self.low = low
        self.high = high

    def sample(self, size=None):
        """
        Generate a sample from the exponential distribution

        Params:
        -------
        size: int, optional (default=None)
            the number of samples to return.  If size=None then a single
            sample is returned.

        Returns:
        -------
        float or np.ndarray (if size >=1)
        """
        return self.rand.uniform(low=self.low, high=self.high, size=size)

In [5]:
class Triangular:
    """
    Convenience class for the triangular distribution.
    packages up distribution parameters, seed and random generator.
    """

    def __init__(self, low, mode, high, random_seed=None):
        """
        Constructor. Accepts and stores parameters of the triangular dist
        and a random seed.

        Params:
        ------
        low: float
            The smallest values that can be sampled

        mode: float
            The most frequently sample value

        high: float
            The highest value that can be sampled

        random_seed: int | SeedSequence, optional (default=None)
            Used with params to create a series of repeatable samples.
        """
        self.rand = np.random.default_rng(seed=random_seed)
        self.low = low
        self.high = high
        self.mode = mode

    def sample(self, size=None):
        """
        Generate one or more samples from the triangular distribution

        Params:
        --------
        size: int
            the number of samples to return.  If size=None then a single
            sample is returned.

        Returns:
        -------
        float or np.ndarray (if size >=1)
        """
        return self.rand.triangular(self.low, self.mode, self.high, size=size)

In [6]:
class Exponential:
    """
    Convenience class for the exponential distribution.
    packages up distribution parameters, seed and random generator.
    """

    def __init__(self, mean, random_seed=None):
        """
        Constructor

        Params:
        ------
        mean: float
            The mean of the exponential distribution

        random_seed: int| SeedSequence, optional (default=None)
            A random seed to reproduce samples.  If set to none then a unique
            sample is created.
        """
        self.rand = np.random.default_rng(seed=random_seed)
        self.mean = mean

    def sample(self, size=None):
        """
        Generate a sample from the exponential distribution

        Params:
        -------
        size: int, optional (default=None)
            the number of samples to return.  If size=None then a single
            sample is returned.

        Returns:
        -------
        float or np.ndarray (if size >=1)
        """
        return self.rand.exponential(self.mean, size=size)

## 3. Clase de experimento

Modificaremos la clase de experimento para incluir una nueva recopilación de resultados para el proceso de enfermería adicional. 

1. Modifique el método __init__ para aceptar parámetros adicionales: `chance_callback`, `nurse_call_low`, `nurse_call_high`. Recuerde incluir los valores predeterminados para estos parámetros.
2. Almacene parámetros en la clase y cree nuevas distribuciones.
3. Agregue variables para admitir el cálculo de KPI al diccionario `results` para `nurse_waiting_times` y `total_nurse_call_duration`.

In [7]:
class Experiment:
    """
    Encapsulates the concept of an experiment 🧪 with the urgent care
    call centre simulation model.

    An Experiment:
    1. Contains a list of parameters that can be left as defaults or varied
    2. Provides a place for the experimentor to record results of a run
    3. Controls the set & streams of pseudo random numbers used in a run.

    """

    def __init__(
        self,
        random_number_set=DEFAULT_RND_SET,
        n_streams=N_STREAMS,
        n_operators=N_OPERATORS,
        mean_iat=MEAN_IAT,
        call_low=CALL_LOW,
        call_mode=CALL_MODE,
        call_high=CALL_HIGH,
        # ######################################################################
        # MODIFICATION: nurse parameters
        n_nurses=N_NURSES,
        chance_callback=CHANCE_CALLBACK,
        nurse_call_low=NURSE_CALL_LOW,
        nurse_call_high=NURSE_CALL_HIGH,
        ########################################################################
    ):
        """
        The init method sets up our defaults.
        """
        # sampling
        self.random_number_set = random_number_set
        self.n_streams = n_streams

        # store parameters for the run of the model
        self.n_operators = n_operators
        self.mean_iat = mean_iat
        self.call_low = call_low
        self.call_mode = call_mode
        self.call_high = call_high

        # resources: we must init resources after an Environment is created.
        # But we will store a placeholder for transparency
        self.operators = None

        # ######################################################################
        # MODIFICATION: nurse parameters
        self.n_nurses = n_nurses
        self.chance_callback = chance_callback
        self.nurse_call_low = nurse_call_low
        self.nurse_call_high = nurse_call_high

        # nurse resources placeholder
        self.nurses = None
        # ######################################################################

        # initialise results to zero
        self.init_results_variables()

        # initialise sampling objects
        self.init_sampling()

    def set_random_no_set(self, random_number_set):
        """
        Controls the random sampling
        Parameters:
        ----------
        random_number_set: int
            Used to control the set of pseudo random numbers used by
            the distributions in the simulation.
        """
        self.random_number_set = random_number_set
        self.init_sampling()

    def init_sampling(self):
        """
        Create the distributions used by the model and initialise
        the random seeds of each.
        """
        # produce n non-overlapping streams
        seed_sequence = np.random.SeedSequence(self.random_number_set)
        self.seeds = seed_sequence.spawn(self.n_streams)

        # create distributions

        # call inter-arrival times
        self.arrival_dist = Exponential(
            self.mean_iat, random_seed=self.seeds[0]
        )

        # duration of call triage
        self.call_dist = Triangular(
            self.call_low,
            self.call_mode,
            self.call_high,
            random_seed=self.seeds[1],
        )

        # ######################################################################
        # MODIFICATION create the callback and nurse consultation distributions
        self.callback_dist = Bernoulli(
            self.chance_callback, random_seed=self.seeds[2]
        )

        self.nurse_dist = Uniform(
            self.nurse_call_low,
            self.nurse_call_high,
            random_seed=self.seeds[3],
        )
        # ######################################################################

    def init_results_variables(self):
        """
        Initialise all of the experiment variables used in results
        collection.  This method is called at the start of each run
        of the model
        """
        # variable used to store results of experiment
        self.results = {}
        self.results["waiting_times"] = []

        # total operator usage time for utilisation calculation.
        self.results["total_call_duration"] = 0.0

        # ######################################################################
        # MODIFICATION: nurse sub process results collection
        self.results["nurse_waiting_times"] = []
        self.results["total_nurse_call_duration"] = 0.0
        # ######################################################################

## 4. Código de modelo modificado

Modificaremos el código del modelo y la lógica que ya hemos desarrollado para incluir una consulta de enfermería para una proporción de las personas que llaman.  Creamos una nueva función llamada `nurse_consultation` que contiene toda la lógica. También necesitamos modificar la función `service` para que una proporción de las llamadas se envíen al proceso de consulta de enfermería.  

In [8]:
def trace(msg):
    """
    Turing printing of events on and off.

    Params:
    -------
    msg: str
        string to print to screen.
    """
    if TRACE:
        print(msg)

In [9]:
def nurse_consultation(identifier, env, args):
    """
    simulates the wait for an consultation with a nurse on the phone.

    1. request and wait for a nurse resource
    2. phone consultation (uniform)
    3. release nurse and exit system

    """
    trace(f"Patient {identifier} waiting for nurse call back")
    start_nurse_wait = env.now

    # request a nurse
    with args.nurses.request() as req:
        yield req

        # record the waiting time for nurse call back
        nurse_waiting_time = env.now - start_nurse_wait
        args.results["nurse_waiting_times"].append(nurse_waiting_time)

        # sample nurse the duration of the nurse consultation
        nurse_call_duration = args.nurse_dist.sample()

        trace(f"nurse called back patient {identifier} at " + f"{env.now:.3f}")

        # schedule process to begin again after call duration
        yield env.timeout(nurse_call_duration)

        args.results["total_nurse_call_duration"] += nurse_call_duration

        trace(
            f"nurse consultation for {identifier}"
            + f" competed at {env.now:.3f}"
        )

In [10]:
def service(identifier, env, args):
    """
    simulates the service process for a call operator

    1. request and wait for a call operator
    2. phone triage (triangular)
    3. release call operator
    4. a proportion of call continue to nurse consultation

    Params:
    ------
    identifier: int
        A unique identifier for this caller

    env: simpy.Environment
        The current environment the simulation is running in
        We use this to pause and restart the process after a delay.

    args: Experiment
        The settings and input parameters for the current experiment

    """

    # record the time that call entered the queue
    start_wait = env.now

    # request an operator - stored in the Experiment
    with args.operators.request() as req:
        yield req

        # record the waiting time for call to be answered
        waiting_time = env.now - start_wait

        # store the results for an experiment
        args.results["waiting_times"].append(waiting_time)
        trace(f"operator answered call {identifier} at " + f"{env.now:.3f}")

        # the sample distribution is defined by the experiment.
        call_duration = args.call_dist.sample()

        # schedule process to begin again after call_duration
        yield env.timeout(call_duration)

        # update the total call_duration
        args.results["total_call_duration"] += call_duration

        # print out information for patient.
        trace(
            f"call {identifier} ended {env.now:.3f}; "
            + f"waiting time was {waiting_time:.3f}"
        )

    # ##########################################################################
    # MODIFICATION NURSE CALL BACK
    # does nurse need to call back?
    # Note the level of the indented code.
    callback_patient = args.callback_dist.sample()

    if callback_patient:
        env.process(nurse_consultation(identifier, env, args))
    # ##########################################################################

In [11]:
def arrivals_generator(env, args):
    """
    IAT is exponentially distributed

    Parameters:
    ------
    env: simpy.Environment
        The simpy environment for the simulation

    args: Experiment
        The settings and input parameters for the simulation.
    """
    # use itertools as it provides an infinite loop
    # with a counter variable that we can use for unique Ids
    for caller_count in itertools.count(start=1):

        # rhe sample distribution is defined by the experiment.
        inter_arrival_time = args.arrival_dist.sample()
        yield env.timeout(inter_arrival_time)

        trace(f"call arrives at: {env.now:.3f}")

        # create a service process
        env.process(service(caller_count, env, args))

## 🥵 Período de calentamiento

El modelo de call center parte de vacío.  Si el centro de llamadas funciona las 24 horas del día, los 7 días de la semana, entonces es un sistema sin terminación y nuestras estimaciones del tiempo de espera y la utilización del servidor están sesgadas debido al período vacío al inicio de la simulación.  Podemos eliminar este sesgo de inicialización mediante un período de calentamiento.  

Implementaremos un calentamiento a través de un **evento** que ocurre una vez en una sola ejecución del modelo.  El modelo se ejecutará durante el **período de calentamiento + período de recolección de resultados**.  Al final del período de calentamiento, ocurrirá un evento en el que todas las variables del experimento actual se restablecerán (por ejemplo, listas vacías y valores cuantitativos establecidos en 0,0).

> **Nota**: en el momento en que se restablecen los resultados, es probable que haya recursos (operadores de llamadas y enfermeras) en uso. El resultado es que trasladamos parte del tiempo de uso de recursos desde el período de preparación hasta la recopilación de resultados. No es gran cosa, pero existe la posibilidad de que el tiempo de uso de recursos sea ligeramente mayor que el tiempo programado.


In [12]:
def warmup_complete(warm_up_period, env, args):
    """
    End of warm-up period event. Used to reset results collection variables.

    Parameters:
    ----------
    warm_up_period: float
        Duration of warm-up period in simultion time units

    env: simpy.Environment
        The simpy environment

    args: Experiment
        The simulation experiment that contains the results being collected.
    """
    yield env.timeout(warm_up_period)
    trace(f"{env.now:.2f}: Warm up complete.")
    
    args.init_results_variables()

## 5. Funciones de contenedor de modelo

Modificaciones a realizar a la función `single_run`:

1. Agregue un parámetro de calentamiento llamado `wu_period`
1. Crear y las enfermeras recursos para el experimento.
2. Programe el proceso `warm_up_complete`.
3. Una vez completada la simulación, calcule el tiempo medio de espera y la utilización media de las enfermeras.

In [13]:
def single_run(
    experiment, 
    rep=0,
    wu_period=WARM_UP_PERIOD, 
    rc_period=RESULTS_COLLECTION_PERIOD
):
    """
    Perform a single run of the model and return the results

    Parameters:
    -----------

    experiment: Experiment
        The experiment/paramaters to use with model

    rep: int
        The replication number.

    wu_period: float, optional (default=WARM_UP_PERIOD)
        The initial transient period of the simulation
        Results from this period are removed from final computations.

    rc_period: float, optional (default=RESULTS_COLLECTION_PERIOD)
        The run length of the model following warm up where results are
        collected.
    """

    # results dictionary.  Each KPI is a new entry.
    run_results = {}

    # reset all results variables to zero and empty
    experiment.init_results_variables()

    # set random number set to the replication no.
    # this controls sampling for the run.
    experiment.set_random_no_set(rep)

    # environment is (re)created inside single run
    env = simpy.Environment()

    # we create simpy resource here - this has to be after we
    # create the environment object.
    experiment.operators = simpy.Resource(env, capacity=experiment.n_operators)

    # #########################################################################
    # MODIFICATION: create the nurses resource
    experiment.nurses = simpy.Resource(env, capacity=experiment.n_nurses)
    # #########################################################################

    # we pass the experiment to the arrivals generator
    env.process(arrivals_generator(env, experiment))

    # #########################################################################
    # MODIFICATON: add warm-up period event
    env.process(warmup_complete(wu_period, env, experiment))

    # run for warm-up + results collection period
    env.run(until=wu_period + rc_period)
    # #########################################################################

    # end of run results: calculate mean waiting time
    run_results["01_mean_waiting_time"] = np.mean(
        experiment.results["waiting_times"]
    )

    # end of run results: calculate mean operator utilisation
    run_results["02_operator_util"] = (
        experiment.results["total_call_duration"]
        / (rc_period * experiment.n_operators)
    ) * 100.0

    # #########################################################################
    # MODIFICATION: summary results for nurse process

    # end of run results: nurse waiting time
    run_results["03_mean_nurse_waiting_time"] = np.mean(
        experiment.results["nurse_waiting_times"]
    )

    # end of run results: calculate mean nurse utilisation
    run_results["04_nurse_util"] = (
        experiment.results["total_nurse_call_duration"]
        / (rc_period * experiment.n_nurses)
    ) * 100.0

    # #########################################################################

    # return the results from the run of the model
    return run_results

In [14]:
def multiple_replications(
    experiment,
    wu_period=WARM_UP_PERIOD,
    rc_period=RESULTS_COLLECTION_PERIOD,
    n_reps=5,
):
    """
    Perform multiple replications of the model.

    Params:
    ------
    experiment: Experiment
        The experiment/paramaters to use with model

    rc_period: float, optional (default=DEFAULT_RESULTS_COLLECTION_PERIOD)
        results collection period.
        the number of minutes to run the model to collect results

    n_reps: int, optional (default=5)
        Number of independent replications to run.

    Returns:
    --------
    pandas.DataFrame
    """

    # loop over single run to generate results dicts in a python list.
    results = [
        single_run(experiment, rep, wu_period, rc_period) 
        for rep in range(n_reps)
    ]

    # format and return results in a dataframe
    df_results = pd.DataFrame(results)
    df_results.index = np.arange(1, len(df_results) + 1)
    df_results.index.name = "rep"
    return df_results

In [15]:
scenario = Experiment(n_nurses=15, nurse_call_high=30.0)
results = multiple_replications(scenario, wu_period=50.0)
results.describe().round(1).T

,count,mean,std,min,25%,50%,75%,max
01_mean_waiting_time,5.0,3.2,0.8,1.9,3.0,3.2,3.9,4.0
02_operator_util,5.0,94.2,1.3,92.1,94.1,94.4,94.5,95.8
03_mean_nurse_waiting_time,5.0,3.0,1.7,1.6,1.7,2.0,4.7,5.0
04_nurse_util,5.0,88.3,3.6,83.7,86.9,87.9,89.5,93.6
